# HEAT1: does short-term heat transfer between correlated stocks?

Deep models (BiLSTM, CNN-ResNet, Transformer) read the last hour of a stock and of its 5 most
correlated peers, and predict its market-relative return over the next 30 minutes, 1, 2 and 4 hours.
Two controls, same model and budget: **no peers** (the stock's own reversal only) and **5 random
peers** (E12's control). Transfer = how much the peers cut the prediction error, in nats. The
**heat map** says which stock's moves feed which, and the **visual report** draws all of it.

Data: the thesis 5-minute SIP bars on Drive (S&P 500, 2016–2026). The rule is registered in
[`configs/heat_v1.yaml`](https://github.com/Gaire-commits/Transfer-entropy-/blob/main/configs/heat_v1.yaml)
(2026-10-20). **Edge** = all of: net of 2 bps above zero on the test days (2024 onward, 99% day-block
interval), beats no peers, beats random peers, positive in both halves, survives false-discovery
control over the 12 cells.

**The order**
1. Setup, then controls.
2. **Check run** (2021 only, never the test days): proves the pipeline works on the real data.
3. **Tune** the deep models on the *validation* days (2022–2023). Never the test days.
4. **Register** the tuned settings as `heat_v2` (before any test scoring), then run it.
5. Result and the visual report.

Tuning is optional: you can skip section 4 and run section 5 with the untuned `heat_v1`.

**Before anything else, pick the GPU:** Runtime → Change runtime type → T4 or better (High-RAM if
offered). Changing the runtime type later resets the session, and then Setup must be run again.
Run top to bottom. Don't use "Run all" once you reach section 5: it scores the test days.

The big panel file stays on this machine's disk (it rebuilds in a few minutes); only the small result
files go to Drive. Every cell that runs a script ends by flushing Drive (`Drive synced.`) with a time
limit, so a slow upload can never freeze the notebook. If the runtime dies *during* a script, anything it hadn't finished is lost; the
architectures already scored are safe, and re-running the cell picks up from them.

## 1. Setup

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
    !pip install -q pyyaml
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO)
sys.path.insert(0, REPO)
for name in [m for m in sys.modules if m == "heat" or m.startswith("heat.")]:
    del sys.modules[name]  # re-running this cell after a pull uses the new code

import html as _html
import torch
from IPython.display import HTML, Markdown, display

DATA = os.environ.get("HEAT_DATA_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/data/bars_sip")
OUT = os.environ.get("HEAT_OUT_DIR", "/content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1")
# The panel is ~420 MB and rebuilds in a few minutes, so it lives on this machine's disk, not Drive:
# one huge upload can hold up the small result files queued behind it. Only results go to Drive.
PANEL = os.environ.get("HEAT_PANEL", "/content/heat_panel.npz")


def sync_drive(timeout=180):
    """Colab uploads to Drive in the background and a runtime restart can lose what hasn't gone up, so flush.
    Never hangs: if the upload takes longer than `timeout` seconds it says so and moves on (the files keep
    uploading while this session lives). If Colab asks you to authorize again after the flush, click through."""
    if not IN_COLAB:
        return
    import threading

    failed = []

    def flush():
        try:
            drive.flush_and_unmount()
        except Exception as exc:  # noqa: BLE001
            failed.append(exc)
            print("(could not flush Drive:", exc, ")")

    t = threading.Thread(target=flush, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive():
        print(f"Drive is still uploading after {timeout}s. Your results are on this machine and keep uploading "
              "in the background; ask Claude to check Drive again in a few minutes.")
        return
    try:
        drive.mount("/content/drive")
        print("Drive NOT confirmed synced." if failed else "Drive synced.")
    except Exception as exc:  # noqa: BLE001
        print("(Drive flushed but could not be remounted:", exc, "- run drive.mount('/content/drive') yourself)")


def download_results(out_dir, name):
    """Zip a run's results (reports, tables, the visual report; not the model weights) and download it
    to your computer. A copy that doesn't depend on Drive."""
    import zipfile
    out_dir = os.path.abspath(out_dir)
    if not os.path.isdir(out_dir):
        print("Nothing to download yet:", out_dir, "does not exist.")
        return
    path = f"/content/{name}.zip" if IN_COLAB else os.path.join(out_dir, f"{name}.zip")
    keep = (".md", ".csv", ".json", ".html", ".yaml")
    n = 0
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out_dir):
            for f in files:
                if f.endswith(keep):
                    z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), out_dir))
                    n += 1
    print(f"{n} files in {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(path)


def show_viz(out_dir, cfg_path="configs/heat_v1.yaml", height=1800):
    """Write <out_dir>/viz.html and show it here (it is also a file you can open full-screen)."""
    from heat import config
    from heat.viz import write_report
    cfg = config.load(cfg_path)
    meta = {"experiment": cfg["experiment"], "cost_bps": cfg["evaluation"]["cost_bps"],
            "level": cfg["evaluation"]["level"], "check": os.path.basename(out_dir) == "check"}
    path = write_report(out_dir, cfg["data"]["universe"], meta, PANEL)
    print("saved", path, "- download it from Drive to open it full-screen in any browser")
    page = open(path).read()
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # IPython suggests IFrame, which can't read a Drive file
        display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:1px solid #8885;border-radius:8px"></iframe>'
                     % (_html.escape(page, quote=True), height)))


if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    tpu = bool(os.environ.get("TPU_NAME") or os.environ.get("COLAB_TPU_ADDR") or os.path.exists("/dev/accel0")
               or os.path.exists("/dev/vfio"))
    print("=" * 78)
    print("NO GPU. " + ("This is a TPU runtime: this code only uses NVIDIA GPUs, so it would run on the CPU, "
                        "many times slower." if tpu else "Training would run on the CPU, many times slower."))
    print("Fix: Runtime -> Change runtime type -> T4 GPU (or L4 / A100), then run this Setup cell again.")
    print("=" * 78)
print("SIP bars found:", os.path.isdir(os.path.join(DATA, "5Min")))
print("outputs:", OUT)

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
SIP bars found: True
outputs: /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1


## 2. Controls (about 5 minutes)

Before real data: on synthetic stocks with a planted leader, the experiment must find the edge and
name the leader in the heat map; with nothing planted it must find nothing. Plus the no-look-ahead
checks, the tuning checks, the visual-report checks and the frozen-config check.

In [2]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} -m pytest -q tests -W ignore

.......................................                                  [100%]
39 passed in 47.57s


## 3. Check run on the real data (inside the training years)

Builds the panel once (every stock's market-relative returns on the thesis grid; a few minutes,
saved to the outputs folder), then a short run that trains to 2019, validates on 2020 and scores
**2021**. It never touches the registered test days, so it's safe to run as often as needed. It shows
whether everything works and gives a feel for the numbers; it is not the result.

In [3]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --data-dir "$DATA" --out-dir "$OUT" --check --panel "$PANEL"
    report = os.path.join(OUT, "check", "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report: the run above stopped with an error. Send a screenshot of its last lines.")
    sync_drive()  # make sure what the script wrote is really on Drive

read 50/503
read 100/503
read 150/503
read 200/503
read 250/503
read 300/503
read 350/503
read 400/503
read 450/503
read 500/503
panel built and saved: 211,380 bars x 503 stocks
peers: k=5, 60 days, monthly
moments: train 4,228,611, val 1,168,018, test 1,157,323
  bilstm/own epoch 1: val 3.35186 (1s)
  bilstm/own epoch 2: val 3.35223 (1s)
  bilstm/own epoch 3: val 3.35206 (1s)
  bilstm/peers epoch 1: val 3.35309 (1s)
  bilstm/peers epoch 2: val 3.35248 (1s)
  bilstm/peers epoch 3: val 3.35285 (1s)
  bilstm/random epoch 1: val 3.35196 (1s)
  bilstm/random epoch 2: val 3.35236 (1s)
  bilstm/random epoch 3: val 3.35215 (1s)
bilstm: scored and saved
  resnet1d/own epoch 1: val 3.39780 (1s)
  resnet1d/own epoch 2: val 3.37575 (1s)
  resnet1d/own epoch 3: val 3.39128 (1s)
  resnet1d/peers epoch 1: val 3.37431 (1s)
  resnet1d/peers epoch 2: val 3.37854 (1s)
  resnet1d/peers epoch 3: val 3.39172 (1s)
  resnet1d/random epoch 1: val 3.36285 (1s)
  resnet1d/random epoch 2: val 3.38785 (1s)
  resn

# heat_v1: transfer between correlated stocks

> **CHECK RUN, not the registered result.** Scored on 2021 (inside the registered training years); the registered test days were not used.

Moments: train 4,228,611, val 1,168,018, test 1,157,323 (scored days: after 2020-12-31). Intervals: 99% day-block. Cost: 2 bps per position.

| arch | hold | G peers (nats) | G peers − random | net bps (peers) | net − no peers | IC | verdict |
|---|---|---|---|---|---|---|---|
| bilstm | 30 min | +0.00008 [-0.00013, +0.00026] | +0.00008 [-0.00009, +0.00023] | -1.92 [-2.44, -1.51] | +0.10 [-0.45, +0.70] | +0.0011 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| bilstm | 60 min | +0.00004 [-0.00020, +0.00030] | +0.00004 [-0.00018, +0.00026] | -1.80 [-2.28, -1.20] | +0.17 [-0.75, +1.18] | +0.0035 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| bilstm | 120 min | -0.00004 [-0.00027, +0.00026] | -0.00002 [-0.00022, +0.00022] | -1.77 [-2.42, -1.00] | -0.18 [-1.11, +1.06] | +0.0033 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| bilstm | 240 min | -0.00043 [-0.00124, +0.00040] | -0.00037 [-0.00110, +0.00039] | -1.71 [-4.09, +1.38] | -0.90 [-6.07, +4.32] | -0.0001 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 30 min | -0.00092 [-0.00173, -0.00020] | -0.00265 [-0.00376, -0.00176] | -1.87 [-2.26, -1.56] | +0.09 [-0.20, +0.36] | +0.0004 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 60 min | -0.00029 [-0.00094, +0.00016] | +0.00018 [-0.00052, +0.00082] | -1.86 [-2.28, -1.56] | +0.05 [-0.40, +0.52] | +0.0016 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 120 min | -0.00142 [-0.00204, -0.00080] | -0.00047 [-0.00121, +0.00013] | -2.19 [-2.84, -1.53] | -0.29 [-1.08, +0.33] | -0.0034 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 240 min | -0.00243 [-0.00342, -0.00147] | -0.00163 [-0.00311, -0.00023] | -1.84 [-3.28, -0.41] | -0.65 [-2.43, +1.00] | +0.0009 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 30 min | -0.00010 [-0.00039, +0.00022] | +0.00001 [-0.00011, +0.00014] | -1.99 [-2.29, -1.64] | -0.12 [-0.69, +0.48] | -0.0034 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 60 min | -0.00033 [-0.00078, +0.00015] | -0.00026 [-0.00065, +0.00019] | -2.06 [-2.69, -1.45] | -0.09 [-0.70, +0.50] | -0.0023 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 120 min | -0.00007 [-0.00028, +0.00015] | +0.00002 [-0.00024, +0.00028] | -2.05 [-2.52, -1.57] | +0.14 [-0.61, +0.77] | -0.0035 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 240 min | +0.00040 [+0.00004, +0.00076] | +0.00004 [-0.00067, +0.00079] | -1.33 [-3.08, +0.69] | +0.86 [-0.55, +2.57] | +0.0025 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |

**Cells with an edge: 0 of 12.** Cells with heat transfer (correlated peers beat random peers and no peers): 0 of 12.

## Where the heat goes

Pairs whose share of the follower's error clears zero by 3 standard errors: **86 of 25,196** (a screening bar across many pairs, not a test).

| arch | hold | source | target | moments | heat share | lower bound |
|---|---|---|---|---|---|---|
| resnet1d | 120 min | CMS | ED | 53 | +0.0621 | +0.0148 |
| resnet1d | 60 min | TXT | WAB | 60 | +0.0493 | +0.0093 |
| resnet1d | 120 min | ABT | ZTS | 82 | +0.0271 | +0.0079 |
| resnet1d | 60 min | MA | GPN | 101 | +0.0420 | +0.0075 |
| resnet1d | 30 min | V | FISV | 113 | +0.0207 | +0.0067 |
| resnet1d | 240 min | TXN | SWKS | 100 | +0.0288 | +0.0060 |
| resnet1d | 240 min | UAL | RCL | 107 | +0.0284 | +0.0053 |
| resnet1d | 120 min | XYZ | GNRC | 84 | +0.0154 | +0.0051 |
| resnet1d | 120 min | JPM | BNY | 87 | +0.0429 | +0.0044 |
| resnet1d | 60 min | ECL | APD | 93 | +0.0222 | +0.0044 |
| resnet1d | 240 min | ADP | PAYX | 105 | +0.0819 | +0.0042 |
| resnet1d | 30 min | VMC | CRH | 74 | +0.0618 | +0.0042 |
| resnet1d | 30 min | CL | KMB | 115 | +0.0348 | +0.0041 |
| transformer | 30 min | DOV | XYL | 103 | +0.0097 | +0.0040 |
| resnet1d | 240 min | IDXX | ILMN | 68 | +0.0215 | +0.0037 |
| resnet1d | 120 min | CRWD | ABNB | 91 | +0.0225 | +0.0035 |
| resnet1d | 240 min | KEY | CFG | 113 | +0.0400 | +0.0035 |
| transformer | 120 min | IVZ | TROW | 60 | +0.0198 | +0.0034 |
| resnet1d | 120 min | JNJ | ABBV | 123 | +0.0234 | +0.0033 |
| resnet1d | 60 min | NTAP | DELL | 96 | +0.0352 | +0.0033 |


Drive is still uploading after 180s. Your results are on this machine and keep uploading in the background; ask Claude to check Drive again in a few minutes.


### The check run, drawn

Transfer by hold, who leads whom (matrix and network), industries leading industries, and who moves
with whom. These are the 2021 numbers from the check run, not the result.

In [4]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(OUT, "check", "cells.csv")) or os.path.exists(os.path.join(OUT, "check", "cells_partial.csv")):
        show_viz(os.path.join(OUT, "check"))
    else:
        print("No check results yet: run the check cell above first.")

No check results yet: run the check cell above first.


**Keep a copy.** Downloads the check run's results to your computer as a zip (reports, tables, the
visual report). It doesn't depend on Drive, so do it even if Drive is slow. Send the zip, or a screenshot,
if Claude can't see the files on Drive.

In [5]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(os.path.join(OUT, "check"), "heat_check_results")

Nothing to download yet: /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1/check does not exist.


## 4. Tune the deep models (validation days only)

Random search with successive halving over learning rate, weight decay, width, depth, dropout, and
each architecture's own knobs (kernel size, attention heads). It only ever sees the training days
(2016–2021) and the validation days (2022–2023); **the test days are never loaded**. The registered
default is always one candidate, and the three variants (no peers, correlated, random) share each
setting, so the peer models get no tuning advantage. The winner is adopted only if it beats the
default by more than the seed-to-seed spread; otherwise the default is kept.

Each architecture is saved to Drive when done and skipped if you run its cell again.

**4a. Quick check (optional, a few minutes):** two trials on 2019–2020, just to see the tuner runs.

In [6]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --check --trials 3 --budgets 1 2 --archs resnet1d
    sync_drive()  # make sure what the script wrote is really on Drive

panel: 211,380 bars x 503 stocks
moments: train 4,228,611, val 1,168,018  (the test days are not used)
  resnet1d trial  0 rung 0 (1 ep): val 3.38366
  resnet1d trial  1 rung 0 (1 ep): val 3.38455
  resnet1d trial  2 rung 0 (1 ep): val 3.37690
  resnet1d trial  2 rung 1 (2 ep): val 3.36894
resnet1d: default 3.42611, tuned 3.43047, spread 0.12306 -> default kept
# Tuning report

Validation days only; the registered test days were not used. Score = mean validation loss over the no-peer, correlated-peer and random-peer models (lower is better).

| arch | default | tuned winner | gain | seed spread | decision |
|---|---|---|---|---|---|
| resnet1d | 3.42611 | 3.43047 | -0.00436 | 0.12306 | default kept |

**resnet1d** chosen: `{'size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.1, 'depth': 3, 'kernel': 3}`

> PARTIAL: architectures still to tune: bilstm, transformer

Drive not mounted, so nothing to flush and unmount.
(could not flush Drive: flush_and_unmount failed )
(Drive flu

**4b. The real tuning**, one architecture per cell. It prints one line per trial, so you can watch it work.

In [7]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --archs bilstm
    sync_drive()  # make sure what the script wrote is really on Drive

panel: 211,380 bars x 503 stocks
moments: train 6,563,337, val 2,336,981  (the test days are not used)
  bilstm trial  0 rung 0 (1 ep): val 1.01044
  bilstm trial  1 rung 0 (1 ep): val 1.01045
  bilstm trial  2 rung 0 (1 ep): val 1.01038
  bilstm trial  3 rung 0 (1 ep): val 1.01060
  bilstm trial  4 rung 0 (1 ep): val 1.01046
  bilstm trial  5 rung 0 (1 ep): val 1.01054
  bilstm trial  6 rung 0 (1 ep): val 1.01045
  bilstm trial  7 rung 0 (1 ep): val 1.01046
  bilstm trial  8 rung 0 (1 ep): val 1.01038
  bilstm trial  9 rung 0 (1 ep): val 1.01049
  bilstm trial 10 rung 0 (1 ep): val 1.01055
  bilstm trial 11 rung 0 (1 ep): val 1.01060
  bilstm trial  2 rung 1 (3 ep): val 1.01038
  bilstm trial  8 rung 1 (3 ep): val 1.01038
  bilstm trial  0 rung 1 (3 ep): val 1.01044
  bilstm trial  6 rung 1 (3 ep): val 1.01044
  bilstm trial  2 rung 2 (9 ep): val 1.01038
  bilstm trial  8 rung 2 (9 ep): val 1.01020
bilstm: default 0.99895, tuned 0.99890, spread 0.02270 -> default kept
# Tuning report


In [8]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --archs resnet1d
    sync_drive()  # make sure what the script wrote is really on Drive

panel: 211,380 bars x 503 stocks
moments: train 6,563,337, val 2,336,981  (the test days are not used)
  resnet1d trial  0 rung 0 (1 ep): val 1.01729
  resnet1d trial  1 rung 0 (1 ep): val 1.01764
  resnet1d trial  2 rung 0 (1 ep): val 1.02377
  resnet1d trial  3 rung 0 (1 ep): val 1.01789
  resnet1d trial  4 rung 0 (1 ep): val 1.01968
  resnet1d trial  5 rung 0 (1 ep): val 1.01707
  resnet1d trial  6 rung 0 (1 ep): val 1.01774
  resnet1d trial  7 rung 0 (1 ep): val 1.01675
  resnet1d trial  8 rung 0 (1 ep): val 1.01714
  resnet1d trial  9 rung 0 (1 ep): val 1.01668
  resnet1d trial 10 rung 0 (1 ep): val 1.01332
  resnet1d trial 11 rung 0 (1 ep): val 1.01593
  resnet1d trial 10 rung 1 (3 ep): val 1.01260
  resnet1d trial 11 rung 1 (3 ep): val 1.01251
  resnet1d trial  9 rung 1 (3 ep): val 1.01346
  resnet1d trial  7 rung 1 (3 ep): val 1.01407
  resnet1d trial 11 rung 2 (9 ep): val 1.01129
  resnet1d trial 10 rung 2 (9 ep): val 1.01111
resnet1d: default 0.99992, tuned 1.00007, spread 0.

In [9]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/tune.py --data-dir "$DATA" --out-dir "$OUT" --panel "$PANEL" --archs transformer
    sync_drive()  # make sure what the script wrote is really on Drive

panel: 211,380 bars x 503 stocks
moments: train 6,563,337, val 2,336,981  (the test days are not used)
  transformer trial  0 rung 0 (1 ep): val 1.01216
  transformer trial  1 rung 0 (1 ep): val 1.01174
  transformer trial  2 rung 0 (1 ep): val 1.01366
  transformer trial  3 rung 0 (1 ep): val 1.01268
  transformer trial  4 rung 0 (1 ep): val 1.01207
  transformer trial  5 rung 0 (1 ep): val 1.01146
  transformer trial  6 rung 0 (1 ep): val 1.01136
  transformer trial  7 rung 0 (1 ep): val 1.01227
  transformer trial  8 rung 0 (1 ep): val 1.01236
  transformer trial  9 rung 0 (1 ep): val 1.01182
  transformer trial 10 rung 0 (1 ep): val 1.01160
  transformer trial 11 rung 0 (1 ep): val 1.01204
  transformer trial  6 rung 1 (3 ep): val 1.01106
  transformer trial  5 rung 1 (3 ep): val 1.01096
  transformer trial 10 rung 1 (3 ep): val 1.01093
  transformer trial  1 rung 1 (3 ep): val 1.01106
  transformer trial 10 rung 2 (9 ep): val 1.01086
  transformer trial  5 rung 2 (9 ep): val 1.010

**4c. What it chose.** When all three are tuned, `tuning/heat_v2.yaml` is the candidate registered
config. **Do not run section 5 yet:** it has to be registered first (committed to the repo with its hash
frozen) so the rule is fixed before the test days are scored. Tell Claude the tuning is done; it reads
the files from Drive, registers `heat_v2`, and pushes. Then re-run Setup (it pulls) and continue.

In [10]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    report = os.path.join(OUT, "tuning", "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
        cand = os.path.join(OUT, "tuning", "heat_v2.yaml")
        print("candidate config written:" , os.path.exists(cand), cand)
    else:
        print("No tuning report yet: run the tuning cells above.")

# Tuning report

Validation days only; the registered test days were not used. Score = mean validation loss over the no-peer, correlated-peer and random-peer models (lower is better).

| arch | default | tuned winner | gain | seed spread | decision |
|---|---|---|---|---|---|
| bilstm | 0.99895 | 0.99890 | +0.00005 | 0.02270 | default kept |
| resnet1d | 0.99992 | 1.00007 | -0.00015 | 0.02238 | default kept |
| transformer | 0.99951 | 0.99937 | +0.00014 | 0.02302 | default kept |

**bilstm** chosen: `{'size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.1, 'depth': 2}`

**resnet1d** chosen: `{'size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.1, 'depth': 3, 'kernel': 3}`

**transformer** chosen: `{'size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.1, 'depth': 2, 'heads': 4}`


candidate config written: True /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1/tuning/heat_v2.yaml


In [11]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(os.path.join(OUT, "tuning"), "heat_tuning_results")  # a copy, independent of Drive

8 files in /content/heat_tuning_results.zip (0.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. The registered run: scores the test days

One architecture per cell, so each fits in a Colab session. Each is saved to Drive as soon as it's
scored, and a scored one is skipped if you run its cell again, so a dropped session just resumes. The
verdict appears once all three are in.

The first cell picks the registered config: `heat_v2` (tuned) if it has been registered in the repo,
otherwise `heat_v1`. It prints which one, so you always know the rule you're running.

In [12]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    import yaml
    CONFIG = "configs/heat_v2.yaml" if os.path.exists("configs/heat_v2.yaml") else "configs/heat_v1.yaml"
    EXP = yaml.safe_load(open(CONFIG))["experiment"]
    REG_OUT = os.path.join(os.path.dirname(OUT), EXP)
    print("registered config:", CONFIG, "->", EXP)
    print("results go to:", REG_OUT)
    if EXP == "heat_v1":
        print("(untuned. If you tuned, tell Claude to register heat_v2, then re-run Setup and this cell.)")

registered config: configs/heat_v1.yaml -> heat_v1
results go to: /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1
(untuned. If you tuned, tell Claude to register heat_v2, then re-run Setup and this cell.)


In [13]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --archs bilstm
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 211,380 bars x 503 stocks
peers: k=5, 60 days, monthly
moments: train 6,563,337, val 2,336,981, test 3,290,308
  bilstm/own epoch 1: val 1.00449 (4s)
  bilstm/own epoch 2: val 1.00457 (4s)
  bilstm/own epoch 3: val 1.00501 (4s)
  bilstm/own epoch 4: val 1.00441 (4s)
  bilstm/own epoch 5: val 1.00474 (4s)
  bilstm/own epoch 6: val 1.00437 (4s)
  bilstm/own epoch 7: val 1.00429 (4s)
  bilstm/own epoch 8: val 1.00465 (4s)
  bilstm/own epoch 9: val 1.00449 (4s)
  bilstm/own epoch 10: val 1.00484 (4s)
  bilstm/own epoch 11: val 1.00477 (4s)
  bilstm/peers epoch 1: val 1.00451 (4s)
  bilstm/peers epoch 2: val 1.00473 (4s)
  bilstm/peers epoch 3: val 1.00510 (4s)
  bilstm/peers epoch 4: val 1.00471 (4s)
  bilstm/peers epoch 5: val 1.00494 (4s)
  bilstm/random epoch 1: val 1.00451 (4s)
  bilstm/random epoch 2: val 1.00479 (4s)
  bilstm/random epoch 3: val 1.00473 (4s)
  bilstm/random epoch 4: val 1.00443 (4s)
  bilstm/random epoch 5: val 1.00485 (4s)
  bilstm/random epoch 6: val 

In [14]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --archs resnet1d
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 211,380 bars x 503 stocks
peers: k=5, 60 days, monthly
moments: train 6,563,337, val 2,336,981, test 3,290,308
  resnet1d/own epoch 1: val 1.00739 (5s)
  resnet1d/own epoch 2: val 1.00734 (4s)
  resnet1d/own epoch 3: val 1.00558 (4s)
  resnet1d/own epoch 4: val 1.01188 (4s)
  resnet1d/own epoch 5: val 1.00526 (4s)
  resnet1d/own epoch 6: val 1.00627 (4s)
  resnet1d/own epoch 7: val 1.00586 (4s)
  resnet1d/own epoch 8: val 1.00509 (4s)
  resnet1d/own epoch 9: val 1.00550 (4s)
  resnet1d/own epoch 10: val 1.00568 (4s)
  resnet1d/own epoch 11: val 1.00522 (4s)
  resnet1d/own epoch 12: val 1.00490 (4s)
  resnet1d/own epoch 13: val 1.00507 (4s)
  resnet1d/own epoch 14: val 1.00489 (4s)
  resnet1d/own epoch 15: val 1.00484 (4s)
  resnet1d/own epoch 16: val 1.00481 (4s)
  resnet1d/own epoch 17: val 1.00600 (4s)
  resnet1d/own epoch 18: val 1.00477 (4s)
  resnet1d/own epoch 19: val 1.00496 (4s)
  resnet1d/own epoch 20: val 1.00480 (4s)
  resnet1d/own epoch 21: val 1.00469 (4s)
  

In [15]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --archs transformer
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 211,380 bars x 503 stocks
peers: k=5, 60 days, monthly
moments: train 6,563,337, val 2,336,981, test 3,290,308
  transformer/own epoch 1: val 1.00601 (7s)
  transformer/own epoch 2: val 1.00569 (6s)
  transformer/own epoch 3: val 1.00519 (6s)
  transformer/own epoch 4: val 1.00529 (6s)
  transformer/own epoch 5: val 1.00484 (6s)
  transformer/own epoch 6: val 1.00476 (6s)
  transformer/own epoch 7: val 1.00469 (6s)
  transformer/own epoch 8: val 1.00484 (6s)
  transformer/own epoch 9: val 1.00470 (6s)
  transformer/own epoch 10: val 1.00483 (6s)
  transformer/own epoch 11: val 1.00501 (6s)
  transformer/peers epoch 1: val 1.00597 (6s)
  transformer/peers epoch 2: val 1.00619 (6s)
  transformer/peers epoch 3: val 1.00527 (6s)
  transformer/peers epoch 4: val 1.00567 (6s)
  transformer/peers epoch 5: val 1.00500 (6s)
  transformer/peers epoch 6: val 1.00558 (6s)
  transformer/peers epoch 7: val 1.00503 (6s)
  transformer/peers epoch 8: val 1.00538 (6s)
  transformer/peers e

## 6. Result

In [16]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    report = os.path.join(REG_OUT, "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report yet: run the cells in section 5 first.")

# heat_v1: transfer between correlated stocks

Moments: train 6,563,337, val 2,336,981, test 3,290,308 (scored days: after 2023-12-31). Intervals: 99% day-block. Cost: 2 bps per position.

| arch | hold | G peers (nats) | G peers − random | net bps (peers) | net − no peers | IC | verdict |
|---|---|---|---|---|---|---|---|
| bilstm | 30 min | -0.00000 [-0.00012, +0.00014] | -0.00004 [-0.00015, +0.00008] | -1.85 [-2.09, -1.64] | +0.31 [+0.11, +0.51] | +0.0085 | NO EDGE (failed: net_positive, beats_random_peers, both_halves_positive, fdr) |
| bilstm | 60 min | -0.00005 [-0.00020, +0.00012] | -0.00015 [-0.00031, +0.00004] | -2.06 [-2.44, -1.71] | +0.27 [+0.03, +0.54] | +0.0054 | NO EDGE (failed: net_positive, beats_random_peers, both_halves_positive, fdr) |
| bilstm | 120 min | -0.00014 [-0.00032, +0.00005] | -0.00023 [-0.00047, +0.00001] | -2.44 [-3.07, -1.82] | +0.26 [-0.18, +0.75] | +0.0019 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| bilstm | 240 min | -0.00011 [-0.00042, +0.00018] | -0.00006 [-0.00041, +0.00025] | -2.13 [-3.05, -1.32] | +0.59 [-0.87, +2.18] | +0.0029 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 30 min | -0.00031 [-0.00059, -0.00003] | -0.00030 [-0.00083, +0.00013] | -1.93 [-2.18, -1.69] | +0.28 [+0.05, +0.51] | +0.0025 | NO EDGE (failed: net_positive, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 60 min | -0.00005 [-0.00041, +0.00032] | -0.00009 [-0.00044, +0.00025] | -1.88 [-2.27, -1.45] | +0.39 [+0.06, +0.72] | +0.0030 | NO EDGE (failed: net_positive, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 120 min | -0.00006 [-0.00052, +0.00039] | -0.00032 [-0.00081, +0.00014] | -1.59 [-2.44, -0.78] | +1.00 [+0.20, +1.91] | +0.0047 | NO EDGE (failed: net_positive, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 240 min | +0.00000 [-0.00070, +0.00076] | +0.00030 [-0.00043, +0.00113] | -0.99 [-2.38, +0.35] | +1.14 [-0.37, +2.53] | +0.0045 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 30 min | -0.00008 [-0.00027, +0.00005] | -0.00001 [-0.00012, +0.00009] | -1.94 [-2.17, -1.72] | +0.06 [-0.18, +0.31] | +0.0022 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 60 min | -0.00004 [-0.00021, +0.00015] | +0.00014 [-0.00003, +0.00031] | -2.16 [-2.67, -1.70] | +0.13 [-0.26, +0.53] | -0.0007 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 120 min | -0.00049 [-0.00098, -0.00006] | -0.00044 [-0.00085, -0.00005] | -2.47 [-3.39, -1.71] | -0.27 [-1.08, +0.58] | -0.0007 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 240 min | -0.00108 [-0.00184, -0.00041] | -0.00088 [-0.00151, -0.00027] | -2.26 [-3.50, -1.13] | +0.69 [-0.94, +2.25] | -0.0001 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |

**Cells with an edge: 0 of 12.** Cells with heat transfer (correlated peers beat random peers and no peers): 0 of 12.

## Where the heat goes

Pairs whose share of the follower's error clears zero by 3 standard errors: **200 of 68,898** (a screening bar across many pairs, not a test).

| arch | hold | source | target | moments | heat share | lower bound |
|---|---|---|---|---|---|---|
| resnet1d | 240 min | BNY | NDAQ | 62 | +0.0111 | +0.0070 |
| resnet1d | 120 min | GLW | KLAC | 50 | +0.0157 | +0.0057 |
| resnet1d | 120 min | HOOD | LYV | 53 | +0.0199 | +0.0056 |
| resnet1d | 120 min | PEP | OTIS | 70 | +0.0101 | +0.0040 |
| transformer | 60 min | VRT | HPE | 58 | +0.0108 | +0.0039 |
| transformer | 240 min | RSG | LDOS | 101 | +0.0065 | +0.0035 |
| resnet1d | 120 min | CVNA | ABNB | 66 | +0.0205 | +0.0031 |
| resnet1d | 60 min | FITB | PRU | 58 | +0.0120 | +0.0030 |
| resnet1d | 30 min | RMD | SYK | 51 | +0.0108 | +0.0028 |
| bilstm | 120 min | HOOD | META | 55 | +0.0128 | +0.0027 |
| resnet1d | 60 min | PAYX | IT | 72 | +0.0069 | +0.0026 |
| resnet1d | 240 min | UPS | TGT | 98 | +0.0138 | +0.0026 |
| transformer | 120 min | NWSA | WBD | 50 | +0.0049 | +0.0025 |
| bilstm | 60 min | PPG | OTIS | 54 | +0.0092 | +0.0024 |
| resnet1d | 60 min | JBL | TRMB | 124 | +0.0090 | +0.0024 |
| transformer | 240 min | CRWD | INTU | 57 | +0.0180 | +0.0024 |
| resnet1d | 60 min | CL | ROL | 82 | +0.0060 | +0.0023 |
| resnet1d | 120 min | PH | PCAR | 73 | +0.0141 | +0.0022 |
| bilstm | 240 min | ZBH | ABT | 52 | +0.0070 | +0.0021 |
| transformer | 60 min | WM | ICE | 63 | +0.0087 | +0.0021 |


### The result, drawn

Use the filters at the top of the report: architecture (or all three averaged), hold, and how many
links to show. Hover any square, arrow or dot for the numbers. Every chart has a table view.

In [17]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(REG_OUT, "cells.csv")) or os.path.exists(os.path.join(REG_OUT, "cells_partial.csv")):
        show_viz(REG_OUT, CONFIG)
    else:
        print("No results yet: run the cells in section 5 first.")

saved /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v1/viz.html - download it from Drive to open it full-screen in any browser


**Keep a copy** of the registered result on your computer (independent of Drive).

In [18]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(REG_OUT, EXP + "_results")

23 files in /content/heat_v1_results.zip (4.4 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>